In [1]:
#Fix for google colab only, not needed for remote server
!pip uninstall -y transformers tokenizers
!pip install -q transformers==4.35.2 tokenizers==0.14.1


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 32.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.17.3 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.17.3 which is incompatible.
accelerate 1.13.0 requires hug

In [2]:

# INSTALL DEPENDENCIES
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate huggingface_hub
!pip install -q opencv-python Pillow matplotlib
!pip install -q scikit-learn nibabel SimpleITK
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.35.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 47.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:

#Clone MedClipSamV2

import os, sys

if not os.path.exists("/content/MedCLIP-SAMv2"):
    !git clone https://github.com/HealthX-Lab/MedCLIP-SAMv2

sys.path.append("/content/MedCLIP-SAMv2")
sys.path.append("/content/MedCLIP-SAMv2/saliency_maps")


Cloning into 'MedCLIP-SAMv2'...
remote: Enumerating objects: 683, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 683 (delta 40), reused 33 (delta 33), pack-reused 625 (from 1)
Receiving objects: 100% (683/683), 13.94 MiB | 35.77 MiB/s, done.
Resolving deltas: 100% (211/211), done.


In [4]:
# Download SAM weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth


In [5]:
#Import CLIP
from transformers.models.clip.modeling_clip import (
    CLIPVisionEmbeddings,
    CLIPTextEmbeddings,
    CLIPEncoder,
    CLIPVisionTransformer
)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [6]:
import torch, cv2, numpy as np
from PIL import Image
from transformers import AutoModel, AutoProcessor, AutoTokenizer
from segment_anything import sam_model_registry, SamPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"

# MedCLIP
model = AutoModel.from_pretrained(
    "chuhac/BiomedCLIP-vit-bert-hf",
    trust_remote_code=True
).to(device)

processor = AutoProcessor.from_pretrained(
    "chuhac/BiomedCLIP-vit-bert-hf",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "chuhac/BiomedCLIP-vit-bert-hf",
    trust_remote_code=True
)

# SAM
sam = sam_model_registry["vit_h"](
    checkpoint="sam_vit_h_4b8939.pth"
).to(device)

sam_predictor = SamPredictor(sam)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_biomed_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/chuhac/BiomedCLIP-vit-bert-hf:
- configuration_biomed_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`model_type` is found in both `text_config_dict` and `text_config` but with different values. The value `text_config_dict["model_type"]` will be used instead.
`model_type` is found in both `vision_config_dict` and `vision_config` but with different values. The value `vision_config_dict["model_type"]` will be used instead.


modeling_biomed_clip.py: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

`model_type` is found in both `text_config_dict` and `text_config` but with different values. The value `text_config_dict["model_type"]` will be used instead.
`model_type` is found in both `vision_config_dict` and `vision_config` but with different values. The value `vision_config_dict["model_type"]` will be used instead.


processing_biomed_clip.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/chuhac/BiomedCLIP-vit-bert-hf:
- processing_biomed_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
!pip install -q grad-cam


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 72.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
from scripts.methods import vision_heatmap_iba
from scripts.plot import visualize_vandt_heatmap


In [15]:
import torch.nn.functional as F

def run_medclipsamv2_slice_no_training(
    image_np,
    text_prompt,
    threshold=0.5
):

    # Prepare image
    image_rgb = np.stack([image_np]*3, axis=-1)
    pil_img = Image.fromarray(image_rgb.astype(np.uint8))

    image_inputs = processor.image_processor(
        images=pil_img,
        return_tensors="pt"
    ).to(device)

    text_inputs = tokenizer(
        [text_prompt],
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    # Forward CLIP (NO gradients)
    with torch.no_grad():
        outputs = model(
            pixel_values=image_inputs["pixel_values"],
            input_ids=text_inputs["input_ids"],
            attention_mask=text_inputs["attention_mask"]
        )

        img_emb = outputs.image_embeds        # (1, D)
        txt_emb = outputs.text_embeds         # (1, D)

    # Global similarity score
    sim = F.cosine_similarity(img_emb, txt_emb).item()

    # Uniform saliency → SAM prompting
    if sim < threshold:
        return np.zeros_like(image_np, dtype=np.uint8)

    sam_predictor.set_image(image_rgb)

    h, w = image_np.shape
    ys, xs = np.meshgrid(
        np.linspace(0, h-1, 10).astype(int),
        np.linspace(0, w-1, 10).astype(int)
    )
    pts = np.stack([ys.flatten(), xs.flatten()], axis=1)
    labels = np.ones(len(pts))

    masks, _, _ = sam_predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )

    return masks[0].astype(np.uint8)


In [10]:
def run_medclipsamv2_volume(volume, text_prompt):
    preds = []
    for z in range(volume.shape[0]):
        preds.append(
            run_medclipsamv2_slice_no_training(volume[z], text_prompt)
        )
    return np.stack(preds, axis=0)


In [16]:
# ==============================
# KAGGLE SETUP
# ==============================
!pip install -q kaggle
!mkdir -p ~/.kaggle

import os

if not os.path.exists("/root/.kaggle/kaggle.json"):
    from google.colab import files
    print("Upload kaggle.json")
    uploaded = files.upload()
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
# ==============================
# DOWNLOAD LGG DATASET
# ==============================
DATASET_PATH = "/content/dataset/lgg-mri-segmentation/kaggle_3m"

if not os.path.exists(DATASET_PATH):
    !kaggle datasets download -d mateuszbuda/lgg-mri-segmentation
    !unzip -q lgg-mri-segmentation.zip -d /content/dataset

print("Dataset path:", DATASET_PATH)
import pandas as pd
from glob import glob

data_csv = os.path.join(DATASET_PATH, "data.csv")
patient_info = pd.read_csv(data_csv)

patient_dirs = sorted(
    glob(os.path.join(DATASET_PATH, "TCGA_*"))
)

print("Patients in CSV:", len(patient_info))
print("Patient folders:", len(patient_dirs))


Dataset path: /content/dataset/lgg-mri-segmentation/kaggle_3m
Patients in CSV: 110
Patient folders: 110


In [12]:
OUTPUT_PATH = "/content/drive/MyDrive/SAM_results_medclipsamv2"
os.makedirs(OUTPUT_PATH, exist_ok=True)


In [13]:
import numpy as np
import nibabel as nib
from PIL import Image
from sklearn.metrics import jaccard_score, f1_score
from scipy.ndimage import zoom

results = []

TEXT_PROMPT = "brain tumor in MRI"


In [17]:
from glob import glob
import numpy as np
from PIL import Image
import os

def load_patient_volume_from_tif(patient_dir):
    image_files = sorted([
        f for f in glob(os.path.join(patient_dir, "*.tif"))
        if f[-5].isdigit() and "_mask" not in f
    ])

    if len(image_files) == 0:
        return None, None

    slices = []
    for f in image_files:
        img = np.array(Image.open(f).convert("L"))
        slices.append(img)

    volume = np.stack(slices, axis=0).astype(np.uint8)
    return volume, image_files

for idx, patient_dir in enumerate(patient_dirs, 1):

    full_id = os.path.basename(patient_dir)
    patient_id = "_".join(full_id.split("_")[:3])

    print(f"\n[{idx}/{len(patient_dirs)}] {patient_id}")


    row = patient_info[patient_info["Patient"] == patient_id]
    if len(row) == 0:
        print("  No demographics, skipping")
        continue

    race   = row.iloc[0]["race"]
    gender = row.iloc[0]["gender"]

    print(f"  Race: {race}, Gender: {gender}")
    # Load volume

    volume, image_files = load_patient_volume_from_tif(patient_dir)
    if volume is None:
        print("  No slices found, skipping")
        continue

    print(f"  Volume shape: {volume.shape}")


    # MedCLIP-SAMv2 inference

    pred_volume = run_medclipsamv2_volume(volume, TEXT_PROMPT)


    out_path = os.path.join(
        OUTPUT_PATH,
        f"{patient_id}_medclipsamv2_predictions.nii.gz"
    )

    nib.save(
        nib.Nifti1Image(pred_volume.astype(np.uint8), np.eye(4)),
        out_path
    )

    print(f"  Saved: {out_path}")


    # Per-slice metrics

    ious, f1s = [], []

    gt_pairs = []
    for img_file in image_files:
        base = os.path.splitext(img_file)[0]
        mask_file = f"{base}_mask.tif"
        if os.path.exists(mask_file):
            gt_pairs.append(mask_file)

    for z, mask_file in enumerate(gt_pairs):
        if z >= pred_volume.shape[0]:
            break

        gt   = np.array(Image.open(mask_file).convert("L"))
        gt   = (gt > 0).astype(np.uint8)
        pred = pred_volume[z]

        if gt.shape != pred.shape:
            scale = (
                gt.shape[0] / pred.shape[0],
                gt.shape[1] / pred.shape[1]
            )
            pred = zoom(pred, scale, order=0).astype(np.uint8)

        gt_sum, pred_sum = gt.sum(), pred.sum()

        if gt_sum == 0 and pred_sum == 0:
            ious.append(1.0)
            f1s.append(1.0)
        elif gt_sum == 0 or pred_sum == 0:
            ious.append(0.0)
            f1s.append(0.0)
        else:
            ious.append(jaccard_score(gt.flatten(), pred.flatten(), zero_division=0))
            f1s.append(f1_score(gt.flatten(), pred.flatten(), zero_division=0))

    mean_iou = np.mean(ious) if ious else 0.0
    mean_f1  = np.mean(f1s)  if f1s  else 0.0

    print(f"  IoU: {mean_iou:.4f}, F1: {mean_f1:.4f}")

    results.append({
        "Patient":         patient_id,
        "Race":            race,
        "Gender":          gender,
        "IoU":             mean_iou,
        "F1":              mean_f1,
        "Num_Slices":      len(ious),
        "Prediction_Path": out_path,
    })



[1/110] TCGA_CS_4941
  Race: 3.0, Gender: 2.0
  Volume shape: (23, 256, 256)
  Slice 1/23
  Slice 2/23
  Slice 3/23
  Slice 4/23
  Slice 5/23
  Slice 6/23
  Slice 7/23
  Slice 8/23
  Slice 9/23
  Slice 10/23
  Slice 11/23
  Slice 12/23
  Slice 13/23
  Slice 14/23
  Slice 15/23
  Slice 16/23
  Slice 17/23
  Slice 18/23
  Slice 19/23
  Slice 20/23
  Slice 21/23
  Slice 22/23
  Slice 23/23
  Saved: /content/drive/MyDrive/SAM_results_medclipsamv2/TCGA_CS_4941_medclipsamv2_predictions.nii.gz
  IoU: 0.6522, F1: 0.6522

[2/110] TCGA_CS_4942
  Race: 2.0, Gender: 1.0
  Volume shape: (20, 256, 256)
  Slice 1/20
  Slice 2/20
  Slice 3/20
  Slice 4/20
  Slice 5/20
  Slice 6/20
  Slice 7/20
  Slice 8/20
  Slice 9/20
  Slice 10/20
  Slice 11/20
  Slice 12/20
  Slice 13/20
  Slice 14/20
  Slice 15/20
  Slice 16/20
  Slice 17/20
  Slice 18/20
  Slice 19/20
  Slice 20/20
  Saved: /content/drive/MyDrive/SAM_results_medclipsamv2/TCGA_CS_4942_medclipsamv2_predictions.nii.gz
  IoU: 0.7000, F1: 0.7000

[3/

In [18]:
# Compute overall summary statistics before saving
results_df = pd.DataFrame(results)

overall_iou = results_df["IoU"].mean() if len(results_df) > 0 else 0.0
overall_f1  = results_df["F1"].mean()  if len(results_df) > 0 else 0.0

print(f"Overall IoU: {overall_iou:.4f}")
print(f"Overall F1 : {overall_f1:.4f}")
print(f"Patients   : {len(results_df)}")


Overall IoU: 0.5964
Overall F1 : 0.5974
Patients   : 119


In [19]:
csv_path = os.path.join(
    OUTPUT_PATH,
    "medclipsamv2_patient_metrics.csv"
)
results_df.to_csv(csv_path, index=False)

print("\nSaved:", csv_path)
race_stats = (
    results_df
    .groupby("Race")[["IoU","F1"]]
    .agg(["mean","std","count"])
    .round(4)
)

print("\n" + "="*70)
print("RACE LEVEL METRICS")
print("="*70)
print(race_stats)

race_stats.to_csv(
    os.path.join(OUTPUT_PATH, "medclipsamv2_race_metrics.csv")
)
gender_stats = (
    results_df
    .groupby("Gender")[["IoU","F1"]]
    .agg(["mean","std","count"])
    .round(4)
)

print("\n" + "="*70)
print("GENDER LEVEL METRICS")
print("="*70)
print(gender_stats)

gender_stats.to_csv(
    os.path.join(OUTPUT_PATH, "medclipsamv2_gender_metrics.csv")
)
summary_path = os.path.join(
    OUTPUT_PATH,
    "medclipsamv2_summary.txt"
)

with open(summary_path, "w") as f:
    f.write("OVERALL METRICS\n")
    f.write("-"*70 + "\n")
    f.write(f"Mean IoU: {overall_iou:.4f}\n")
    f.write(f"Mean F1 : {overall_f1:.4f}\n")
    f.write(f"Patients: {len(results_df)}\n\n")

    f.write("RACE-LEVEL METRICS\n")
    f.write("-"*70 + "\n")
    f.write(race_stats.to_string())
    f.write("\n\n")

    f.write("GENDER-LEVEL METRICS\n")
    f.write("-"*70 + "\n")
    f.write(gender_stats.to_string())
    f.write("\n")

print("Saved summary:", summary_path)
print("Results directory:", OUTPUT_PATH)




Saved: /content/drive/MyDrive/SAM_results_medclipsamv2/medclipsamv2_patient_metrics.csv

RACE LEVEL METRICS
         IoU                    F1              
        mean     std count    mean     std count
Race                                            
2.0   0.5431  0.2702    12  0.5439  0.2685    12
3.0   0.5999  0.1911   105  0.6009  0.1879   105

GENDER LEVEL METRICS
           IoU                    F1              
          mean     std count    mean     std count
Gender                                            
1.0     0.6230  0.1758    59  0.6235  0.1739    59
2.0     0.5678  0.2188    59  0.5693  0.2150    59
Saved summary: /content/drive/MyDrive/SAM_results_medclipsamv2/medclipsamv2_summary.txt
Results directory: /content/drive/MyDrive/SAM_results_medclipsamv2
